# Naming conventions
- PL = Pipeline
- NB = Notebook
- VAR = Variable Library
- SQL = Fabric Database
- LH = Lakehouse
- WH = Warehouse
- WS = Workspace


## Fabric Database Connection

Make sure you create a Fabric Database Connection first Settings-Manage Connection-Cloud Conenction-Fabric database type

# Install the Fabric CLI

as of Pythion 3.12 the Fabric CLI is installed by default

Check out the documentation for all commands

In [ ]:
%pip install ms-fabric-cli  --quiet

In [ ]:
import subprocess
import os
from time import sleep, time
import json
import shutil
import re
import requests
import zipfile
import struct


from io import BytesIO
from zipfile import ZipFile 

## Capacity configuration

In [ ]:
capacity_name = "Trial-Erwin"                              # Which capacity will be used for these workspaces


## Parameters

## Add you workspace names below, which you have created in the previous Labs

In [ ]:
workspace_name_code='WS EDK CODE2'
workspace_name_data='WS EDK DATA2'
workspace_name_config='WS EDK CONFIG2'
database_name='SQL_DEMO'

## CLI Login
We need to login to get permissions to create items on behalf of your own user identity

In [ ]:
# Set environment parameters for Fabric CLI
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token


## Repo Configuration

In [ ]:
#FMD Framework code
##### DO NOT CHANGE UNLESS SPECIFIED OTHERWISE ####
repo_owner = "edkreuk"              # Owner of the repository
repo_name = "FabricAutomation-Workshop"         # Name of the repository
branch = "main"                     #"main" is default                    
folder_prefix = "Labs/Lab 4 Data Integration"
###################################################


In [ ]:

def download_folder_as_zip(repo_owner, repo_name, output_zip, branch="main", folder_to_extract="src",  remove_folder_prefix = ""):
    # Construct the URL for the GitHub API to download the repository as a zip file
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball/{branch}"
    
    # Make a request to the GitHub API
    response = requests.get(url)
    response.raise_for_status()
    
    # Ensure the directory for the output zip file exists
    os.makedirs(os.path.dirname(output_zip), exist_ok=True)
    
    # Create a zip file in memory
    with zipfile.ZipFile(BytesIO(response.content)) as zipf:
        with zipfile.ZipFile(output_zip, 'w') as output_zipf:
            for file_info in zipf.infolist():
                parts = file_info.filename.split('/')
                if  re.sub(r'^.*?/', '/', file_info.filename).startswith(folder_to_extract): 
                    # Extract only the specified folder
                    file_data = zipf.read(file_info.filename)
                    output_zipf.writestr(('/'.join(parts[1:]).replace(remove_folder_prefix, "")), file_data)

def uncompress_zip_to_folder(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    os.remove(zip_path)

def copy_to_tmp(name):
    shutil.rmtree("./builtin/tmp", ignore_errors=True)
    path2zip = "./builtin/src/src.zip"

    prefixes = [
        f"src/{name}"
    ]

    with ZipFile(path2zip) as archive:
        for prefix in prefixes:
            matched_files = [file for file in archive.namelist() if file.startswith(prefix)]
            if matched_files:
                for file in matched_files:
                    archive.extract(file, "./builtin/tmp")
                return f"./builtin/tmp/{prefix}"  # Return only the first matching prefix

    return None  # Nothing found

# ✅ Combine all folders into one zip
download_folder_as_zip(repo_owner, repo_name, output_zip = "./builtin/src/src.zip", branch = branch, folder_to_extract= f"/{folder_prefix}/src", remove_folder_prefix = f"{folder_prefix}/")


mapping_table=[]


In [ ]:
def replace_ids_and_mark_inactive(folder_path, mapping_table):
    """
    Replaces old IDs with new ones in JSON-based files and deactivates activities
    that reference connections not in the target_guids list.

    Parameters:
    - folder_path (str): Path to the folder containing files to process.
    - mapping_table (list): List of dictionaries with 'old_id', 'new_id', and 'environment'.
    - environment_name (str): Current environment name to filter applicable mappings.
    - target_guids (list): List of valid connection GUIDs to retain as active.

    Returns:
    - None. Files are modified in-place.
    """
    def find_externalReferences_in_dict(j):
        externalReferences = {}
        for key, value in j.items():
            if isinstance(value, dict):
                externalReferences.update(find_externalReferences_in_dict(value))
            if key == "externalReferences":
                externalReferences[key] = value
        return externalReferences
    
    for root, _, files in os.walk(folder_path):
        for file_name in files:
            if file_name.endswith(('.py', '.json', '.pbir', '.platform', '.ipynb', '.tmdl')) and not file_name.endswith('report.json'):
                file_path = os.path.join(root, file_name)

                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()

                # Replace IDs
                for mapping in mapping_table:
                    content = content.replace(mapping[f"old_id"], mapping[f"new_id"])

                try:
                    data = json.loads(content)
                except json.JSONDecodeError:
                    continue

                if not data or not data.get("properties") or not data["properties"].get("activities"):
                    continue

                content = json.dumps(data, indent=2)
                with open(file_path, 'w', encoding='utf-8') as file:
                    file.write(content)

In [ ]:
def get_item_id(workspace_name, name, property):
    """
    Retrieves the item ID from a workspace.
    """
    return run_fab_command(f"get /{workspace_name}.Workspace/{name} -f -q {property}", capture_output=True, silently_continue=True)

def get_workspace_id_by_name(workspace_name):
    """
    Retrieves the workspace ID by its display name.
    """
    result = run_fab_command("api -X get workspaces/", capture_output=True, silently_continue=True)
    workspaces = json.loads(result)["text"]["value"]
    normalized_name = workspace_name.strip().lower()
    match = next((w for w in workspaces if w['displayName'].strip().lower() == normalized_name), None)
    return match['id'] if match else None


In [ ]:
# -------------------------------
# FABRIC CLI Utilities
# -------------------------------

def run_fab_command(command, capture_output=False, silently_continue=False, raw_output=False):
    """
    Executes a Fabric CLI command with optional output capture and error handling.
    """
    result = subprocess.run(["fab", "-c", command], capture_output=capture_output, text=True)
    if not silently_continue and (result.returncode > 0 or result.stderr):
        raise Exception(f"Error running fab command. exit_code: '{result.returncode}'; stderr: '{result}'")
    if capture_output:
        return result if raw_output else result.stdout.strip()
    return None

def copy_to_tmp(name):
    """
    Extracts item files from a ZIP archive to a temporary directory,
    including all subfolders under the specified path.
    Checks src/{name} first; if not found, checks src/business_domain/{name}.
    Returns the path for the first match only.
    """
    shutil.rmtree("./builtin/tmp", ignore_errors=True)
    path2zip = "./builtin/src/src.zip"

    prefixes = [
        f"src/{name}",
        f"src/business_domain/{name}"
    ]

    with ZipFile(path2zip) as archive:
        for prefix in prefixes:
            matched_files = [file for file in archive.namelist() if file.startswith(prefix)]
            if matched_files:
                for file in matched_files:
                    archive.extract(file, "./builtin/tmp")
                return f"./builtin/tmp/{prefix}"  # Return only the first matching prefix

    return None  # Nothing found

def deploy_item(workspace_name,name,mapping_table):
    """
    Deploys an item (Notebook, Lakehouse, DataPipeline) into a workspace.
    Handles ID replacement, description assignment, and updates mapping and task logs.

    Parameters:
    - workspace (dict): Workspace configuration including name.
    - name (str): Name of the item to deploy.
    - mapping_table (list): List to store ID mappings.
    - environment_name (str): Target environment name.
    - connection_list (list): List of valid connection GUIDs.
    - tasks (list): List to store task execution logs.
    - lakehouse_schema_enabled (bool): Flag to enable schema creation for lakehouses.
    - child (str, optional): Child item name if applicable.
    - it (dict, optional): Item metadata including old ID.
    """
    start = time()
    print("\n#############################################")
    print(f"Deploying in {workspace_name}: {name}")

    tmp_path = copy_to_tmp(name)


    cli_parameter = ''

    if "Notebook" in name:
        cli_parameter += " --format .py"
        result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f {cli_parameter}",capture_output=True, silently_continue=True)
    
    elif "DataPipeline" in name:
        try:
            print(f"Creating or updating DataPipeline: {name}")
            replace_ids_and_mark_inactive(tmp_path, mapping_table)
            result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f",capture_output=True, silently_continue=True)
            print(f"✅ {name} Created/Imported'")
        except Exception as e:
            print(f"❌ Failed to create DataPipeline: {e}")

    elif "VariableLibrary" in name:   
        
        try:
            print(f"Creating or updating VariableLibrary: {name}")
            result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f",capture_output=True, silently_continue=True)
            print(f"✅ {name} Created/Imported'")
        except Exception as e:
            print(f"❌ Failed to create VariableLibrary: {e}")

    
    elif "Environment" in name:   #Not working yet, import is giving error back
        try:
            print(f"Creating or updating Environment: {name}")
            result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f",capture_output=True, silently_continue=True)
            print(f"✅ {name} Created/Imported'")
        except Exception as e:
            print(f"❌ Failed to create Environment: {e}")
    elif "SQLDatabase" in name:  
        tmp_path = copy_to_tmp('SQL_FMD_DEMO_FRAMEWORK.SQLDatabase')  #This is the folder in Github repo
        try:
            print(f"Creating or updating SQLDatabase: {name}")
            result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f",capture_output=True, silently_continue=True)
            print(f"✅ {name} Created/Imported'")
        except Exception as e:
            raise RuntimeError(f"❌ Failed to create database: {e}")
    new_id = get_item_id(workspace_name, name, 'id')
    if it:
        mapping_table.append({"old_id": it["id"],"new_id": new_id})


        

In [ ]:
item_deployment=[
    {
        "name": "NB_FMD_PROCESSING_PARALLEL_MAIN.Notebook",
        "id": "5d142f2d-00c2-8c02-4a4f-36e0cf32a3fb",
        "type": "Notebook"
    },
        {
        "name": "NB_FMD_LOAD_BRONZE_SILVER.Notebook",
        "id": "435ab8df-6841-b4c1-48d8-8bb641a013cc",
        "type": "Notebook"
    },
        {
        "name": "NB_FMD_LOAD_LANDING_BRONZE.Notebook",
        "id": "e0f39cbe-8e7e-be18-445a-b1ca952dcd00",
        "type": "Notebook"
    },
        {
        "name": "PL_FMD_LDZ_COPY_FROM_ONELAKE_TABLES_01.DataPipeline",
        "id": "39f89b03-42a1-be06-4205-257cf99ccf4c",
        "type": "DataPipeline"
        }
        ,
        {
        "name": "PL_FMD_LOAD_LANDINGZONE.DataPipeline",
        "id": "44eb45e8-db7c-ade0-44aa-5a2941e99633",
        "type": "DataPipeline"
    },
        {
        "name": "PL_FMD_LOAD_SILVER.DataPipeline",
        "id": "a1084b6c-dbbb-8633-474f-69ce52e23fdd",
        "type": "DataPipeline"
    },
        {
        "name": "PL_FMD_LOAD_BRONZE.DataPipeline",
        "id": "c2614891-28b7-a603-4a7c-c9e1d28707d6",
        "type": "DataPipeline"
    },
        {
        "name": "PL_FMD_LOAD_ALL.DataPipeline",
        "id": "9a83751a-2bd1-ab49-4c0a-1b081e6d5b96",
        "type": "DataPipeline"
    },
        {
        "name": "VAR_CONFIG_FMD.VariableLibrary",
        "id": "b1ff126e-70d7-9596-4480-a6c1d19144e7",
        "type": "VariableLibrary"
    },
        {
        "name": "SQL_FMD_DEMO_FRAMEWORK.SQLDatabase",
        "id": "dcb978d3-b3cb-449b-9439-c14fcdda674f",
        "type": "SQLDatabase"
    }
    ]

## Create workspaces

In [ ]:
!fab mkdir {workspace_name_code}.workspace -P capacityName="{capacity_name}"
!fab mkdir {workspace_name_data}.workspace -P capacityName="{capacity_name}"
!fab mkdir {workspace_name_config}.workspace -P capacityName="{capacity_name}"

## Create Lakehouse

In [ ]:
#Lakehouse name to be used do not change

lakehouses=['LH_Data_Landingzone','LH_Bronze_Layer','LH_Silver_Layer']
for lakehouse_name in lakehouses:
    !fab create {workspace_name_data}.Workspace/{lakehouse_name}.lakehouse -P enableschemas=true

## Create Fabric database

In [ ]:
!fab create {workspace_name_config}.Workspace/{database_name}.SQLDatabase


In [ ]:
workspace_id=get_workspace_id_by_name(workspace_name_code)
mapping_table.append({"old_id": "00000000-0000-0000-0000-000000000000","new_id": workspace_id})


In [ ]:
for it in item_deployment:
    if it['type'] in ('SQLDatabase'):

        name = database_name
        type = it["type"]
        name=name+'.'+type
        deploy_item(workspace_name_config,name,mapping_table)

### Deploy Notebooks from Repo

In [ ]:
for it in item_deployment:
    if it['type'] in ('Notebook'):

        name = it["name"]
        type = it["type"]
        deploy_item(workspace_name_code,name,mapping_table)

### Deploy Variable Library from Repo

In [ ]:
for it in item_deployment:
    if it['type'] in ('VariableLibrary'):

        name = it["name"]
        type = it["type"]
        deploy_item(workspace_name_code,name,mapping_table)

### Deploy Datapipelines from Repo

In [ ]:
for it in item_deployment:
    if it['type'] in ('DataPipeline'):

        name = it["name"]
        type = it["type"]
        deploy_item(workspace_name_code,name,mapping_table)

## Below are the settings which you can use to update the Variable Library

In [ ]:
fmd_fabric_db_connectionstring = get_item_id(workspace_name_config,  database_name+'.SQLDatabase', 'properties.serverFqdn')
fmd_fabric_db_name = get_item_id(workspace_name_config,  database_name+'.SQLDatabase', 'properties.databaseName')
fmd_config_database_guid=get_item_id(workspace_name_config, database_name+'.SQLDatabase', 'id')
fmd_config_workspace_guid=get_workspace_id_by_name(workspace_name_config)

LH_Data_Landingzone_guid= get_item_id(workspace_name_data, 'LH_Data_Landingzone.Lakehouse', 'id')
LH_Bronze_Layer_guid= get_item_id(workspace_name_data, 'LH_Bronze_Layer.Lakehouse', 'id')
LH_Silver_Layer_guid= get_item_id(workspace_name_data, 'LH_Silver_Layer.Lakehouse', 'id')
data_workspace=get_workspace_id_by_name(workspace_name_data)


print("fmd_fabric_db_connectionstring : " + fmd_fabric_db_connectionstring  )  
print("fmd_fabric_db_name : " + fmd_fabric_db_name  )  
print("fmd_config_database_guid : " + fmd_config_database_guid  )  
print("fmd_config_workspace_guid : " + fmd_config_workspace_guid  ) 
print("fmd_fabric_db_connection : Get the id from the connection"   )   

print("LH_Data_Landingzone : " + LH_Data_Landingzone_guid  )  
print("LH_Data_Landingzone : " + LH_Bronze_Layer_guid  )  
print("LH_Data_Landingzone : " + LH_Silver_Layer_guid  )  
print("data_workspace : " + data_workspace  )  


